# 04 - Topic Modelling with BERTopic (Objective 4)
Pipeline: sentence embeddings -> UMAP -> HDBSCAN -> c-TF-IDF.

**Key design choice:** reviews are split into **sentences** first, so every sentence gets one topic. This makes topic-level sentiment (notebook 07) much cleaner than assigning one topic to a whole review.

In [ ]:
%pip install -q bertopic sentence-transformers umap-learn hdbscan gensim scikit-learn pandas pyarrow

In [ ]:
import re
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

df = pd.read_parquet("data/processed/reviews_clean.parquet")

## 1. Split reviews into sentences

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 3]

sent = df[["review_id", "hotel_id", "review_date", "text_light"]].copy()
sent["sentence"] = sent["text_light"].apply(split_sentences)
sent = sent.explode("sentence").dropna(subset=["sentence"]).drop(columns="text_light").reset_index(drop=True)
sent["sentence_id"] = sent.index
docs = sent["sentence"].tolist()
print(len(df), "reviews ->", len(docs), "sentences")

## 2. Create embeddings once (the slow step) and reuse them for every experiment

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(docs, show_progress_bar=True, batch_size=64)
embeddings.shape

## 3. Experiment with settings
We try several `min_cluster_size` values and compare:
- **number of topics**, **outlier share** (topic -1; lower is better)
- **coherence (c_v)** - higher is better
- **diversity** - share of unique words across topics, higher is better

In [ ]:
def run_bertopic(min_cluster_size=30, n_neighbors=15, n_components=5):
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0,
                      metric="cosine", random_state=42)
    hdb = HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean",
                  cluster_selection_method="eom", prediction_data=True)
    vec = CountVectorizer(stop_words="english", min_df=5)            # unigrams -> simple coherence
    model = BERTopic(embedding_model=embedder, umap_model=umap_model, hdbscan_model=hdb,
                     vectorizer_model=vec, top_n_words=10, verbose=False)
    topics, _ = model.fit_transform(docs, embeddings)
    return model, topics

def evaluate(model, topics):
    ids = [t for t in model.get_topics() if t != -1]
    words = [[w for w, _ in model.get_topic(t)] for t in ids]
    tokens = [d.lower().split() for d in docs]
    tokens = [[w.strip(".,!?") for w in d] for d in tokens]
    dictionary = Dictionary(tokens)
    words = [[w for w in ws if w in dictionary.token2id] for ws in words]
    coherence = CoherenceModel(topics=words, texts=tokens, dictionary=dictionary, coherence="c_v").get_coherence()
    diversity = len({w for ws in words for w in ws}) / max(1, sum(len(ws) for ws in words))
    return {"n_topics": len(ids), "outlier_share": np.mean(np.array(topics) == -1),
            "coherence": coherence, "diversity": diversity}

results = []
for mcs in [15, 30, 50]:
    m, t = run_bertopic(min_cluster_size=mcs)
    results.append({"min_cluster_size": mcs, **evaluate(m, t)})
pd.DataFrame(results).round(3)

## 4. Fit the final model and reduce to a manageable number of topics

In [ ]:
topic_model, topics = run_bertopic(min_cluster_size=30)
topic_model.reduce_topics(docs, nr_topics=12)          # merge similar topics
topics = topic_model.topics_
topic_model.get_topic_info()[["Topic", "Count", "Name"]]

In [ ]:
# read example sentences of each topic - the most important quality check
for t in sorted(set(topics) - {-1})[:6]:
    print(f"\nTopic {t}:", [w for w, _ in topic_model.get_topic(t)][:6])
    for s in pd.Series(docs)[np.array(topics) == t].sample(2, random_state=1):
        print("   -", s)

## 5. Map topics to business themes
Automatic keyword matching gives a first draft. **Review and fix the mapping manually.**

In [ ]:
THEMES = {
    "staff": ["staff", "reception", "friendly", "rude", "helpful", "manager", "service"],
    "cleanliness": ["clean", "dirty", "spotless", "filthy", "bathroom", "smell", "tidy"],
    "food": ["breakfast", "food", "restaurant", "delicious", "dinner", "meal", "coffee"],
    "location": ["location", "metro", "centre", "area", "walk", "close", "far", "noisy"],
    "room": ["room", "bed", "view", "spacious", "tiny", "comfortable", "air"],
    "price": ["price", "value", "expensive", "money", "cost", "worth"],
    "facilities": ["pool", "gym", "spa", "wifi", "lift", "facilities", "parking"],
}

def theme_for_topic(topic_id):
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    scores = {theme: sum(w in kw for w in words) for theme, kw in THEMES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "other"

topic_to_theme = {t: theme_for_topic(t) for t in set(topics) if t != -1}
topic_to_theme[-1] = "outlier"
print(topic_to_theme)

# manual overrides if the automatic mapping is wrong, e.g.:
# topic_to_theme[3] = "food"

In [ ]:
sent["topic_id"] = topics
sent["theme"] = sent["topic_id"].map(topic_to_theme)
from pathlib import Path
Path("models").mkdir(exist_ok=True)
sent.to_parquet("data/processed/sentences_topics.parquet", index=False)
topic_model.save("models/bertopic_model", serialization="safetensors", save_ctfidf=True,
                 save_embedding_model="sentence-transformers/all-MiniLM-L6-v2")
sent["theme"].value_counts()

In [ ]:
# optional visuals
topic_model.visualize_barchart(top_n_topics=8)